# Cellpose 3D processing with full control
This pipeline is processing the 3D stack as a serie of 2D images to stitch

In [1]:
import imageio
from skimage.transform import rescale
import numpy as np
from skimage import io
import napari
from os import listdir
from os.path import isfile, join
import read_roi
import os
from roifile import roiread
import matplotlib.pyplot as plt


## define function to intake images and ROI file and crop and sum first 2 channels and save it

In [2]:

# Function to crop and sum the images
def process_images(image_directory, roi_file):
    # Read ROI coordinates
    rois = roiread(roi_file)
    roi_coords = np.array(rois.coordinates())

    # Extracting the bounding box coordinates
    x1, y1 = np.min(roi_coords, axis=0).astype(int)
    x2, y2 = np.max(roi_coords, axis=0).astype(int)

    # Process all images in the directory
    processed_images = []
    original_file_names = []  # Capturing the original file names
    for image_file in os.listdir(image_directory):
        if image_file.endswith('.tif'):
            image_path = os.path.join(image_directory, image_file)
            print(image_path)
            image = io.imread(image_path)
            dimensions = image.shape
            print("Dimensions of the image:", dimensions)
            cropped_image = image[:, y1:y2, x1:x2, :]
            dimensions_crop = cropped_image.shape
            print("Dimensions of the cropped image:", dimensions_crop)
            summed_image = np.sum(cropped_image[:, :, :, :2], axis=-1)
            processed_images.append(summed_image)
            original_file_names.append(image_file)  # Adding the original file name to the list

    return processed_images, original_file_names  # Returning both the processed images and the original file names

# Function to save and preview the processed images
def save_and_preview_images(processed_images, original_file_names, save_directory):
    for idx, raw_image in enumerate(processed_images):
        # Get the original file name without the extension
        original_file_name_without_extension = os.path.splitext(original_file_names[idx])[0]
        
        # Append "cropped" and add the extension back
        new_file_name = original_file_name_without_extension + '_cropped.tif'
        
        # Save the image using the new file name
        save_path = os.path.join(save_directory, new_file_name)
        io.imsave(save_path, raw_image.astype(np.uint16))
        
        # Preview the first slice of the image
        plt.imshow(raw_image[150, :, :], cmap='gray')
        plt.title(f'Preview of Processed Image {idx} (First Slice)')
        plt.show()



## run cropping and sum function 

In [3]:

# Directory of images and ROI file (update with actual paths)
image_directory = r'U:\Scientific Data\RG-AS04-Data01\Oded_Mayseless\imaging_data\spinSR\20250120_CaMPARI_rpp25b\tiff_files\reg_files\merged_files'
roi_file = r'C:\Oded_data\campari_pipeline\OB_ROI.roi'

# Directory to save processed images 
save_directory = image_directory + os.sep + 'output' 
if not os.path.exists(save_directory):
    os.mkdir(save_directory)

# Process the images
processed_images, original_file_names = process_images(image_directory, roi_file)

# Save and preview the processed images
#save_and_preview_images(processed_images, original_file_names, save_directory)


U:\Scientific Data\RG-AS04-Data01\Oded_Mayseless\imaging_data\spinSR\20250120_CaMPARI_rpp25b\tiff_files\reg_files\merged_files\ref_20250120CaMPARIrpp25b-Arg100uM60sec10secUV01_1_01_warp_m0g80c4e1e-1x52r3.nrrd_merged.tif
Dimensions of the image: (301, 1152, 1152, 3)
Dimensions of the cropped image: (301, 198, 433, 3)
U:\Scientific Data\RG-AS04-Data01\Oded_Mayseless\imaging_data\spinSR\20250120_CaMPARI_rpp25b\tiff_files\reg_files\merged_files\ref_20250120CaMPARIrpp25b-Arg100uM60sec10secUV02_2_01_warp_m0g80c4e1e-1x52r3.nrrd_merged.tif
Dimensions of the image: (301, 1152, 1152, 3)
Dimensions of the cropped image: (301, 198, 433, 3)
U:\Scientific Data\RG-AS04-Data01\Oded_Mayseless\imaging_data\spinSR\20250120_CaMPARI_rpp25b\tiff_files\reg_files\merged_files\ref_20250120CaMPARIrpp25b-Arg100uM60sec10secUV03_3_01_warp_m0g80c4e1e-1x52r3.nrrd_merged.tif
Dimensions of the image: (301, 1152, 1152, 3)
Dimensions of the cropped image: (301, 198, 433, 3)
U:\Scientific Data\RG-AS04-Data01\Oded_Maysele

## Start segmentation processing

In [4]:
from cellpose import models, io
#import imageio
#from skimage.transform import rescale
#import napari
from skimage import img_as_ubyte
from pathlib import Path
import os
import glob
import numpy as np


In [5]:
# Helper function to run the cellpose model
def run_cellpose_model_3D(
                       img,
                       do_3D = True,
                       model = 'Cyto2',
                       pretrained_model = None,
                       cellprob_th = -20.0,
                       diameter = 12,
                       channels=[0,0],
                       use_GPU = True
                      ):
    logger = io.logger_setup()
    # Run cellpose model inference
    if pretrained_model:
        model = models.CellposeModel(gpu=use_GPU, pretrained_model=pretrained_model)
    else:
        model = models.CellposeModel(gpu=use_GPU, model_type=model)
    mask, flows, styles = model.eval(img, 
                                     channels=channels, 
                                     diameter=diameter, 
                                     do_3D=do_3D, 
                                     #net_avg=False, 
                                     #augment=False, 
                                     cellprob_threshold=cellprob_th,
                                    )
        
    return mask, flows, styles

def run_cellpose_model_2Dstitch(
                       img,
                       do_3D = False,
                       model = 'cyto2',
                       pretrained_model = None,
                       cellprob_th = -10.0,
                       Flow_threshold = 0.0,
                       Stitch_threshold = -10.0,
                       diameter = 12,
                       channels=[0,0],
                       use_GPU = True
                      ):
    
    logger = io.logger_setup()
    # Run cellpose model inference
    if pretrained_model:
        model = models.CellposeModel(gpu=use_GPU, pretrained_model=pretrained_model)
    else:
        model = models.CellposeModel(gpu=use_GPU, model_type=model)
    mask, flows, styles = model.eval(img, 
                                     do_3D=do_3D, 
                                     channels=channels, 
                                     diameter=diameter, 
                                     #net_avg=False, 
                                     #augment=False, 
                                     cellprob_threshold=cellprob_th,
                                     flow_threshold=Flow_threshold,
                                     stitch_threshold=Stitch_threshold, 
                                    )
        
    return mask, flows, styles

# Process the whole zstack, 3D wise

In [6]:
import gc
gc.collect()

61944

In [7]:
diameter = 12
process_as_3D = True
channel = [0,0]
# Use this approach to run a pretrained model
pretrained_model = 'C:/Oded_data/campari_pipeline/CaMPARI_cells_cyto2_new'
image_directory = r'U:\Scientific Data\RG-AS04-Data01\Oded_Mayseless\imaging_data\spinSR\20250120_CaMPARI_rpp25b\tiff_files\reg_files\merged_files'
# Directory to save processed images 
save_directory = image_directory + os.sep + 'output' 
if not os.path.exists(save_directory):
    os.mkdir(save_directory)

output_directory = save_directory
#'U:/Home/maysel0000/imaging_data/processed_data/20240116_CaMPARI_100uMCitricAcid_60sec_10secUV/tiff_files/reg_files/merged_files/output/'

# Loop over both processed images and original file names
for raw_image, original_file_name in zip(processed_images, original_file_names):
    if process_as_3D:
        mask, flows, _ = run_cellpose_model_3D(raw_image, diameter=diameter, pretrained_model=pretrained_model)
    else:
        mask, flows, _ = run_cellpose_model_2Dstitch(raw_image, diameter=diameter, pretrained_model=pretrained_model)

    # Construct the save path using the original file name
    save_path = os.path.join(output_directory, original_file_name.replace('.tif', '_mask.tif'))

    # Save the masks using the constructed save path
    io.masks_flows_to_seg(raw_image, mask, flows, diameter, save_path, channel)
    io.save_masks(raw_image, mask, flows, save_path, png=False, tif=True)


2025-01-23 10:41:52,898 [INFO] WRITING LOG OUTPUT TO C:\Users\maysel0000\.cellpose\run.log
2025-01-23 10:41:52,898 [INFO] 
cellpose version: 	2.2.2 
platform:       	win32 
python version: 	3.9.17 
torch version:  	1.12.0
2025-01-23 10:41:52,898 [INFO] >>>> loading model C:/Oded_data/campari_pipeline/CaMPARI_cells_cyto2_new
2025-01-23 10:41:53,072 [INFO] ** TORCH CUDA version installed and working. **
2025-01-23 10:41:53,072 [INFO] >>>> using GPU
2025-01-23 10:41:55,383 [INFO] >>>> model diam_mean =  30.000 (ROIs rescaled to this size during training)
2025-01-23 10:41:55,384 [INFO] >>>> model diam_labels =  10.697 (mean diameter of training ROIs)
2025-01-23 10:41:55,445 [INFO] multi-stack tiff read in as having 301 planes 1 channels
2025-01-23 10:41:57,249 [INFO] running YX: 301 planes of size (198, 433)
2025-01-23 10:42:47,503 [INFO] 100%|##########| 151/151 [00:49<00:00,  3.04it/s]
2025-01-23 10:42:49,846 [INFO] running ZY: 198 planes of size (301, 433)
2025-01-23 10:44:04,587 [INFO]

## expand masks to original dimensions

In [21]:
# Function to expand masks to original dimensions
def expand_masks_to_original_dimensions(original_image_directory, mask_directory, roi_file, save_directory):
    # Read ROI coordinates
    rois = roiread(roi_file)
    roi_coords = np.array(rois.coordinates())
    x1, y1 = np.min(roi_coords, axis=0).astype(int)
    x2, y2 = np.max(roi_coords, axis=0).astype(int)

    for original_image_file in os.listdir(original_image_directory):
        if original_image_file.endswith('.tif'):
            # Define the save path for the expanded mask
            save_path = os.path.join(save_directory, original_image_file.replace('.tif', '_expanded_mask.tif'))
            
            # Skip processing if the file already exists
            if os.path.exists(save_path):
                print(f"File already exists, skipping: {save_path}")
                continue
                
            # Load the original image to get its dimensions
            original_image_path = os.path.join(original_image_directory, original_image_file)
            original_image = io.imread(original_image_path)

            # Create an empty mask with the same dimensions as the original image but with one channel
            expanded_mask_shape = (original_image.shape[0], 1, original_image.shape[2], original_image.shape[3])
            expanded_mask = np.zeros(expanded_mask_shape, dtype=np.uint16)

            # Load the corresponding mask
            mask_file_name = original_image_file.replace('.tif', '_mask_cp_masks.tif')
            mask_path = os.path.join(mask_directory, mask_file_name)
            cropped_mask = io.imread(mask_path)

            # Place the cropped mask into the expanded mask
            expanded_mask[:, 0, y1:y2, x1:x2] = cropped_mask

            # Save the expanded mask
            io.imsave(save_path, expanded_mask)

            
            
            
# Example usage

save_directory_expanded_masks = save_directory + os.sep + 'expanded_masks' 
if not os.path.exists(save_directory_expanded_masks):
    os.mkdir(save_directory_expanded_masks)

#save_directory_expanded_masks = 'U:/Home/maysel0000/imaging_data/processed_data/20240116_CaMPARI_100uMCitricAcid_60sec_10secUV/tiff_files/reg_files/merged_files/output/expanded_masks/'
#save_directory_masks ='U:/Home/maysel0000/imaging_data/processed_data/20240116_CaMPARI_100uMCitricAcid_60sec_10secUV/tiff_files/reg_files/merged_files/output/'
expand_masks_to_original_dimensions(image_directory, save_directory, roi_file, save_directory_expanded_masks)


File already exists, skipping: U:\Scientific Data\RG-AS04-Data01\Oded_Mayseless\imaging_data\spinSR\20250120_CaMPARI\tiff_files\reg_files\merged_files\output\expanded_masks\ref_20250120CaMPARIrpp25b-Arg100uM60sec10secUV01_1_01_warp_m0g80c4e1e-1x52r3.nrrd_merged_expanded_mask.tif
File already exists, skipping: U:\Scientific Data\RG-AS04-Data01\Oded_Mayseless\imaging_data\spinSR\20250120_CaMPARI\tiff_files\reg_files\merged_files\output\expanded_masks\ref_20250120CaMPARIrpp25b-Arg100uM60sec10secUV02_2_01_warp_m0g80c4e1e-1x52r3.nrrd_merged_expanded_mask.tif
File already exists, skipping: U:\Scientific Data\RG-AS04-Data01\Oded_Mayseless\imaging_data\spinSR\20250120_CaMPARI\tiff_files\reg_files\merged_files\output\expanded_masks\ref_20250120CaMPARIrpp25b-Arg100uM60sec10secUV03_3_01_warp_m0g80c4e1e-1x52r3.nrrd_merged_expanded_mask.tif
File already exists, skipping: U:\Scientific Data\RG-AS04-Data01\Oded_Mayseless\imaging_data\spinSR\20250120_CaMPARI\tiff_files\reg_files\merged_files\output\ex

100%|████████████████████████████████████████████████████████████████████████████████| 903/903 [00:42<00:00, 21.08it/s]


2025-01-23 23:41:54,702 [INFO] reading tiff with 301 planes


100%|███████████████████████████████████████████████████████████████████████████████| 301/301 [00:01<00:00, 288.24it/s]


2025-01-23 23:42:05,357 [INFO] reading tiff with 903 planes


...

100%|███████████████████████████████████████████████████████████████████████████████| 301/301 [00:01<00:00, 243.92it/s]


2025-01-24 00:25:00,701 [INFO] reading tiff with 903 planes


100%|████████████████████████████████████████████████████████████████████████████████| 903/903 [00:41<00:00, 21.68it/s]


2025-01-24 00:25:42,609 [INFO] reading tiff with 301 planes


100%|███████████████████████████████████████████████████████████████████████████████| 301/301 [00:01<00:00, 256.84it/s]


In [20]:
image_directory = r'U:\Scientific Data\RG-AS04-Data01\Oded_Mayseless\imaging_data\spinSR\20250120_CaMPARI\tiff_files\reg_files\merged_files'
save_directory = image_directory + os.sep + 'output'

In [22]:
#filtering cells using regionprops and extract intensity infromation from the raw image
# Import necessary libraries
import tifffile as tiff
import numpy as np
from scipy.ndimage import label
from skimage import measure
from scipy.spatial import distance
from scipy.ndimage import zoom
from scipy.spatial import cKDTree
from scipy.ndimage import distance_transform_edt
import time
import csv
import re
import os
import pandas as pd
from skimage.measure import regionprops
import tifffile as tif

def extract_odorant(filename):
    pattern = r'(?i)CaMPARI([A-Za-z0-9]+)'  # Added (?i) for case-insensitive matching
    match = re.search(pattern, filename)
    return match.group(1) if match else None
def processed_output_exists(filename, output_folder):
    # Check if the corresponding CSV output exists in the output folder.
    odorant_name = extract_odorant(filename)
    csv_filename = os.path.join(output_folder, f'{odorant_name}_cell_distances_intensities.csv')
    return os.path.exists(csv_filename)

def filter_cells_by_distance(cell_segmentation_image, labeled_image, raw_data_image, filename, pixel_to_micron, output_folder,distance):
    start_time = time.time()
    selected_labels = {1: "mdG2", 2: "maG", 3: "mdG6", 4: "dG", 5: "vmG", 6: "lG", 7: "dlG", 8: "vpG"}
#    selected_labels = {1: "mdG2", 2: "maG"}
    
    # Get unique cell labels
    unique_cell_labels = np.unique(cell_segmentation_image)
    unique_cell_labels = unique_cell_labels[unique_cell_labels != 0]  # Exclude background label if present

    # DataFrame to save cell centroid, distance to each label, and intensity measurements
    cell_data_df = pd.DataFrame(index=unique_cell_labels)
    cell_data_df.index.name = 'cell_label_id'
    
    # Dictionary for filtered cells images
    filtered_cells_images_by_label = {label_name: np.zeros_like(cell_segmentation_image) for label_name in selected_labels.values()}
    
    for label_value, label_name in selected_labels.items():
        region_mask = (labeled_image == label_value)
        distance_transform = distance_transform_edt(~region_mask)
        boundary_coords = np.argwhere(distance_transform == 1)
        boundary_coords_scaled = boundary_coords * [pixel_to_micron['z'], pixel_to_micron['y'], pixel_to_micron['x']]
        boundary_tree = cKDTree(boundary_coords_scaled)

        # Calculate region properties for centroids and volumes
        props = regionprops(label_image=cell_segmentation_image)
        centroids = []
        volumes = []
        for prop in props:
            if prop['label'] in unique_cell_labels:
                centroids.append(prop.centroid)
                volumes.append(prop.area)  # In 3D, 'area' corresponds to the volume
        
        # Ensure there are centroids found before proceeding
        if not centroids:
            print(f"No centroids found for file: {filename}. Skipping processing for this file.")
            return None, None
        
        centroids = np.array(centroids)
        volumes = np.array(volumes)
        
        # Scale centroids
        centroids_scaled = centroids * [pixel_to_micron['z'], pixel_to_micron['y'], pixel_to_micron['x']]
        
        # Save the scaled centroids to the DataFrame
        cell_data_df['centroid_x'] = centroids_scaled[:, 2]  # x-coordinate
        cell_data_df['centroid_y'] = centroids_scaled[:, 1]  # y-coordinate
        cell_data_df['centroid_z'] = centroids_scaled[:, 0]  # z-coordinate
        
        # Save volumes to the DataFrame
        cell_data_df['volume'] = volumes * np.prod(list(pixel_to_micron.values()))  # Adjust volume by pixel to micron scaling
        min_distances, _ = boundary_tree.query(centroids_scaled, k=1)

        # Save distances to current label
        cell_data_df[label_name] = pd.Series(min_distances, index=[prop['label'] for prop in props if prop['label'] in unique_cell_labels])

        # Filter cells by distance and update filtered_cells_images_by_label
        filtered_cells_labels = [props[i]['label'] for i in np.arange(len(props)) if min_distances[i] < distance]
        for i, cell_label in enumerate(filtered_cells_labels):
            filtered_cells_images_by_label[label_name][cell_segmentation_image == cell_label] = cell_label

        # Collect intensity statistics for each channel
        for channel in range(3):
            single_channel_intensity_image = raw_data_image[:, channel, :, :]
            props_intensity = regionprops(label_image=cell_segmentation_image, intensity_image=single_channel_intensity_image)
            mean_intensities = [prop['mean_intensity'] for prop in props_intensity]
            max_intensities = [prop['max_intensity'] for prop in props_intensity]
            min_intensities = [prop['min_intensity'] for prop in props_intensity]
            cell_data_df[f'mean_intensity_channel_{channel}'] = mean_intensities
            cell_data_df[f'max_intensity_channel_{channel}'] = max_intensities
            cell_data_df[f'min_intensity_channel_{channel}'] = min_intensities

    # Save to CSV
    odorant_name = extract_odorant(filename)
    csv_filename = os.path.join(output_folder, f'{odorant_name}_cell_distances_intensities.csv')
    cell_data_df.to_csv(csv_filename)

    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"Filtering cells took {elapsed_time:.2f} seconds.")

    return filtered_cells_images_by_label, cell_data_df  # Return both values





2025-01-24 00:25:55,712 [INFO] Note: NumExpr detected 20 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 8.
2025-01-24 00:25:55,712 [INFO] NumExpr defaulting to 8 threads.


In [23]:

def extract_odorant(filename):
    # Define a pattern to match everything after "CaMPARI" up to the next delimiter
    pattern = r'(?i)CaMPARI([A-Za-z0-9\-]+)'  # Adjusted to include hyphens
    match = re.search(pattern, filename)
    return match.group(1) if match else None


def process_data(raw_data_folder, cell_segmentation_folder, labeled_image_path, output_folder, pixel_to_micron, distance):
    # Read the labeled image
    labeled_image = tif.imread(labeled_image_path)

    # Iterate through the raw data files
    for filename in os.listdir(raw_data_folder):
        if filename.endswith(".tif"):
            # Check if the file has already been processed
            if processed_output_exists(filename, output_folder):
                print(f'Skipping processing for {filename} as it has already been processed.')
                continue

            print('Processing', filename)
            raw_data_path = os.path.join(raw_data_folder, filename)
            cell_segmentation_filename = filename.replace('_merged.tif', '_merged_expanded_mask.tif')
            cell_segmentation_path = os.path.join(cell_segmentation_folder, cell_segmentation_filename)

            # Read the raw data and cell segmentation images
            raw_data_image = tif.imread(raw_data_path)
            cell_segmentation_image = tif.imread(cell_segmentation_path)
            cell_segmentation_image = np.squeeze(cell_segmentation_image)

            # Filter cells by distance
            filtered_cells_images_by_label, cell_data_df = filter_cells_by_distance(cell_segmentation_image, labeled_image, raw_data_image, filename, pixel_to_micron, output_folder, distance)

    print("Processing completed.")
    return filtered_cells_images_by_label, cell_data_df, cell_segmentation_image, raw_data_image, labeled_image


# Define the paths and parameters
#image_directory = r'U:\Scientific Data\RG-AS04-Data01\Oded_Mayseless\imaging_data\spinSR\20250120_CaMPARI_rpp25b\tiff_files\reg_files\merged_files'
raw_data_folder = image_directory

cell_segmentation_folder = raw_data_folder + os.sep + "output/expanded_masks/"
#cell_segmentation_folder = "U:/Home/maysel0000/imaging_data/processed_data/20240116_CaMPARI_100uMCitricAcid_60sec_10secUV/tiff_files/reg_files/merged_files/output/expanded_masks/"
output_folder = cell_segmentation_folder  + os.sep + "output"
if not os.path.exists(output_folder):
    os.mkdir(output_folder)
#output_folder = "U:/Home/maysel0000/imaging_data/processed_data/20240116_CaMPARI_100uMCitricAcid_60sec_10secUV/tiff_files/reg_files/merged_files/output/expanded_masks/output"
labeled_image_path = "C:\Oded_data\campari_pipeline\glomeruli_as_onech.tif"
pixel_to_micron = {'x': 0.4333, 'y': 0.4333, 'z': 1.0}
distance = 5
# Run the process
filtered_cells_images_by_label, cell_data_df, cell_segmentation_image, raw_data_image, labeled_image = process_data(raw_data_folder, cell_segmentation_folder, labeled_image_path, output_folder, pixel_to_micron,distance)


Processing ref_20250120CaMPARIrpp25b-Arg100uM60sec10secUV01_1_01_warp_m0g80c4e1e-1x52r3.nrrd_merged.tif
Filtering cells took 933.00 seconds.
Processing ref_20250120CaMPARIrpp25b-Arg100uM60sec10secUV02_2_01_warp_m0g80c4e1e-1x52r3.nrrd_merged.tif
Filtering cells took 965.24 seconds.
Processing ref_20250120CaMPARIrpp25b-Arg100uM60sec10secUV03_3_01_warp_m0g80c4e1e-1x52r3.nrrd_merged.tif
Filtering cells took 1033.67 seconds.
Processing ref_20250120CaMPARIrpp25b-Arg100uM60sec10secUV04_4_01_warp_m0g80c4e1e-1x52r3.nrrd_merged.tif
Filtering cells took 962.50 seconds.
Processing ref_20250120CaMPARIrpp25b-Arg100uM60sec10secUV05_5_01_warp_m0g80c4e1e-1x52r3.nrrd_merged.tif
Filtering cells took 961.20 seconds.
Processing ref_20250120CaMPARIrpp25b-Arg100uM60sec10secUV06_6_01_warp_m0g80c4e1e-1x52r3.nrrd_merged.tif
Filtering cells took 966.42 seconds.
Processing ref_20250120CaMPARIrpp25b-ATP500uM60sec10secUV01_7_01_warp_m0g80c4e1e-1x52r3.nrrd_merged.tif
Filtering cells took 971.33 seconds.
Processing r